# Tutorial 06: Adjoint-Consistent Boundary Conditions

This tutorial demonstrates how boundary condition consistency affects MFG solutions when **stall points occur at domain boundaries**.

## Mathematical Background

At the Boltzmann-Gibbs equilibrium, the **total flux** must vanish at reflecting boundaries:

$$J = -\frac{\sigma^2}{2}\nabla m + m\,\alpha^* = 0$$

Standard Neumann BC ($\partial u / \partial n = 0$) implies zero drift at the wall. But when the stall point is AT the boundary, equilibrium requires $\nabla u \cdot n \neq 0$. The fix:

$$\frac{\partial u}{\partial n} = -\frac{\sigma^2}{2} \frac{\partial \ln m}{\partial n}$$

This **Robin BC** couples HJB to the FP density gradient, maintaining consistency.

## Setup

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from mfgarchon import MFGProblem
from mfgarchon.alg.numerical.coupling.fixed_point_iterator import FixedPointIterator
from mfgarchon.alg.numerical.fp_solvers.fp_fdm import FPFDMSolver
from mfgarchon.alg.numerical.hjb_solvers.hjb_fdm import HJBFDMSolver
from mfgarchon.core import MFGComponents
from mfgarchon.core.hamiltonian import QuadraticControlCost, SeparableHamiltonian
from mfgarchon.geometry import TensorProductGrid
from mfgarchon.geometry.boundary import (
    AdjointConsistentProvider,
    BCSegment,
    BCType,
    BoundaryConditions,
    neumann_bc,
)

NX, NT, T, SIGMA = 50, 50, 1.0, 0.2
MAX_ITERATIONS, TOLERANCE = 30, 1e-6

## Step 1: Boundary Stall Problem\n\nAgents want to reach $x = 0$ (left boundary). Terminal cost $u_T(x) = x^2$ has minimum at boundary.

In [ ]:
def create_lq_components():
    H = SeparableHamiltonian(
        control_cost=QuadraticControlCost(control_cost=1.0),
        coupling=lambda m: 0.5 * m,
        coupling_dm=lambda m: 0.5,
    )
    return MFGComponents(
        hamiltonian=H,
        m_initial=lambda x: np.exp(-20 * (x - 0.3) ** 2),
        u_terminal=lambda x: x**2,  # Min at x=0 (boundary stall!)
    )

## Step 2: Standard Neumann BC\n\nClassical: $\partial u / \partial n = 0$ on both boundaries.

In [ ]:
geo_std = TensorProductGrid(bounds=[(0.0, 1.0)], Nx_points=[NX + 1], boundary_conditions=neumann_bc(dimension=1))
prob_std = MFGProblem(geometry=geo_std, T=T, Nt=NT, sigma=SIGMA, components=create_lq_components())

iter_std = FixedPointIterator(problem=prob_std, hjb_solver=HJBFDMSolver(prob_std), fp_solver=FPFDMSolver(prob_std))
result_std = iter_std.solve(max_iterations=MAX_ITERATIONS, tolerance=TOLERANCE, verbose=True)
print(f"Standard: {result_std.iterations} iters, error={result_std.max_error:.6e}")

## Step 3: Adjoint-Consistent BC

Replace Neumann with Robin via `AdjointConsistentProvider`. The provider computes $g = -\sigma^2/2 \cdot \partial \ln m / \partial n$ from the current density each Picard iteration.

In [ ]:
bc_ac = BoundaryConditions(
    segments=[
        BCSegment(
            name="left_ac",
            bc_type=BCType.ROBIN,
            alpha=0.0,
            beta=1.0,
            value=AdjointConsistentProvider(side="left", diffusion=SIGMA),
            boundary="x_min",
        ),
        BCSegment(
            name="right_ac",
            bc_type=BCType.ROBIN,
            alpha=0.0,
            beta=1.0,
            value=AdjointConsistentProvider(side="right", diffusion=SIGMA),
            boundary="x_max",
        ),
    ],
    dimension=1,
)

geo_ac = TensorProductGrid(bounds=[(0.0, 1.0)], Nx_points=[NX + 1], boundary_conditions=bc_ac)
prob_ac = MFGProblem(geometry=geo_ac, T=T, Nt=NT, sigma=SIGMA, components=create_lq_components())

iter_ac = FixedPointIterator(problem=prob_ac, hjb_solver=HJBFDMSolver(prob_ac), fp_solver=FPFDMSolver(prob_ac))
result_ac = iter_ac.solve(max_iterations=MAX_ITERATIONS, tolerance=TOLERANCE, verbose=True)
print(f"Adjoint: {result_ac.iterations} iters, error={result_ac.max_error:.6e}")

## Step 4: Compare

In [ ]:
print(f"Standard:  {result_std.iterations} iters, error = {result_std.max_error:.6e}")
print(f"Adjoint:   {result_ac.iterations} iters, error = {result_ac.max_error:.6e}")
if result_ac.max_error > 0:
    print(f"Improvement: {result_std.max_error / result_ac.max_error:.2f}x")

## Step 5: Visualize

In [ ]:
x = np.linspace(0, 1, prob_std.geometry.get_grid_shape()[0])
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Standard vs Adjoint-Consistent BC", fontsize=14, fontweight="bold")

axes[0, 0].plot(x, result_std.M[-1], "b-", label="Standard", lw=2)
axes[0, 0].plot(x, result_ac.M[-1], "r--", label="Adjoint", lw=2)
axes[0, 0].set_title("Final Density m(T,x)")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(x, result_std.U[-1], "b-", label="Standard", lw=2)
axes[0, 1].plot(x, result_ac.U[-1], "r--", label="Adjoint", lw=2)
axes[0, 1].set_title("Final Value U(T,x)")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(x, result_std.M[-1] - result_ac.M[-1], "g-", lw=2)
axes[1, 0].axhline(0, color="k", ls="--", alpha=0.5)
axes[1, 0].set_title("Density Difference")
axes[1, 0].grid(alpha=0.3)

if len(result_std.error_history_U) > 0:
    axes[1, 1].semilogy(result_std.error_history_U, "b-o", label="Standard", ms=4)
    axes[1, 1].semilogy(result_ac.error_history_U, "r-s", label="Adjoint", ms=4)
    axes[1, 1].set_title("Convergence")
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

| BC Mode | When to Use |
|:--------|:------------|
| **Standard Neumann** | Interior stall, periodic BC, general |
| **Adjoint-Consistent Robin** | Boundary stall, reflecting walls, high accuracy |

**Pattern**: `AdjointConsistentProvider` in `BCSegment.value` + automatic resolution by `FixedPointIterator`.